In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Tours

In [ ]:
def total_tours(data1, data2, tag='PSRC Region'):
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    if int(model_year) > 2023:
        if tag == 'PSRC Region':
            Tour_2_total *= TOUR_FACTOR_PSRC_2023
        elif tag == 'BKR':
            Tour_2_total *= TOUR_FACTOR_BKR_2023

    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = Tour_1_total
    tpp[f'{survey_year}Survey'] = Tour_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.0f}',
        f'{survey_year}Survey': '{:,.0f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.0f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
total_tours(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_tours(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour per Person

In [ ]:
def tour_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    if int(model_year) > 2023:
        if tag == 'PSRC Region':
            Tour_2_total *= TOUR_FACTOR_PSRC_2023
        elif tag == 'BKR':
            Tour_2_total *= TOUR_FACTOR_BKR_2023

    ##Tours per person
    tpp1 = Tour_1_total / Person_1_total
    tpp2 = Tour_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.2f}',
        f'{survey_year}Survey': '{:,.2f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.2f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
tour_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tour_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Share by Purpose

In [ ]:
def pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Purpose
    tour_1_total = get_total(data1['Tour']['toexpfac'])
    tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'pdpurp', 'Tour Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Purpose',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Tours by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Tours', 
                      xaxis_title='Tour Purpose',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True),
                      autosize=True)
                    #   width=950)
    fig.update_yaxes(ticksuffix='%')
    fig.show(config={"responsive": True})

In [ ]:
pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_tour_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')


## Tour Share by Mode

In [ ]:
def pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Mode
    tour_1_total = get_total(data1['Tour']['toexpfac'])
    tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'tmodetp', 'Tour Mode')
    ptbp = ptbp.loc[mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Mode',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Tour Share by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tour Share', 
                      xaxis_title='Tour Mode',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [ ]:
pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tours per Person by Purpose

In [ ]:
def tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'pdpurp', 'Tour Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.2f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.2f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.2f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Purpose',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()    

In [ ]:
tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tours per Person by Mode

In [ ]:
def tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'tmodetp', 'Tour Mode')
    tpbp = tpbp.loc[mode_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.2f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.2f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.2f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Mode',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()    

In [ ]:
tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Distance by Purpose

In [ ]:
def tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.2f}',
        f'Average Tour Length ({name2})': '{:,.2f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Distance by Mode

In [ ]:
def tour_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    #Average Distance by Tour Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.2f}',
        f'Average Tour Length ({name2})': '{:,.2f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Travel Time by Purpose

In [ ]:
def tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.2f}',
        f'Average Tour Travel Time ({name2})': '{:,.2f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.2f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Tour Travel Time by Mode

In [ ]:
def tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
tours_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Subtour Purpose Share

In [ ]:
def subtours_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                     feature='dpurp', feature_label='Purpose', feature_order=[]):
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    #Subtour Purpose Share
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'parent', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'parent', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    subtour_ok_1 = tour_ok_1[tour_ok_1['parent']>0].copy(deep=True)
    subtour_ok_2 = tour_ok_2[tour_ok_2['parent']>0].copy(deep=True)
    stpurpose1 = subtour_ok_1[[feature,'toexpfac']].groupby(feature).sum()['toexpfac']
    stpurpose2 = subtour_ok_2[[feature,'toexpfac']].groupby(feature).sum()['toexpfac']
    subpurposeshare1 = stpurpose1 / Tour_1_total * 100
    subpurposeshare2 = stpurpose2 / Tour_2_total * 100
    spsdf = pd.DataFrame()
    difference = subpurposeshare1 - subpurposeshare2
    subpurposeshare1 = subpurposeshare1.sort_index()
    spsdf[name1 + ' Share (%)'] = subpurposeshare1
    spsdf[name1 + ' # of Subtours'] = stpurpose1
    subpurposeshare2 = subpurposeshare2.sort_index()
    spsdf[name2 + ' Share (%)'] = subpurposeshare2
    spsdf[name2 + ' # of Subtours'] = stpurpose2
    spsdf = get_differences(spsdf, name1 + ' Share (%)', name2 + ' Share (%)', 3)
    spsdf = recode_index(spsdf, feature, feature_label)
    spsdf = spsdf.loc[feature_order, [name1 + ' Share (%)',
                                      name1 + ' # of Subtours',
                                      name2 + ' Share (%)',
                                      name2 + ' # of Subtours',
                                      f'Difference ({name1} Share (%) - {name2} Share (%))']]
    spsdf = spsdf.loc[pdpurp_cat.values()]
    # display table
    def pct_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.3f}%'
    def cnt_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.0f}'
    display(spsdf.style.format({
        f'{name1} Share (%)': pct_fmt,
        f'{name1} # of Subtours': cnt_fmt,
        f'{name2} Share (%)': pct_fmt,
        f'{name2} # of Subtours': cnt_fmt,
        f'Difference ({name1} Share (%) - {name2} Share (%))': pct_fmt
    }))
    # bar plot
    fig = px.bar(
        spsdf.reset_index(),
        x=feature_label,
        y=[f'{name1} Share (%)', f'{name2} Share (%)'],
        barmode='group',
        title=f'Subtour {feature_label} Share ({tag})'
    )
    fig.update_layout(yaxis_title='Share (%)', 
                      xaxis_title=feature_label, 
                      showlegend=True,
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%', gridcolor='lightgrey', showgrid=True)
    fig.show()

In [ ]:
subtours_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                 feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

In [ ]:
subtours_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                     feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

## Number of Stops

In [ ]:
def num_stops_all_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ## Number of Stops for all Purposes
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    # number of stops can be retrieved from tripsh1 and tripsh2, which aggregate the number of stops from origin and destination
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    # calculate the percentage of each number of stops
    intermediate_stops1 = data1['Tour'].groupby('all_stops')['toexpfac'].sum().reset_index()
    intermediate_stops2 = data2['Tour_cloned'].groupby('all_stops')['toexpfac'].sum().reset_index()
    intermediate_stops1['percentage'] = intermediate_stops1['toexpfac'] / intermediate_stops1['toexpfac'].sum() * 100
    intermediate_stops2['percentage'] = intermediate_stops2['toexpfac'] / intermediate_stops2['toexpfac'].sum() * 100
    # compare the first 10 stops
    imstp1 = intermediate_stops1[intermediate_stops1['all_stops']<=10].copy(deep=True)
    imstp2 = intermediate_stops2[intermediate_stops2['all_stops']<=10].copy(deep=True)
    imstp1 = imstp1.set_index('all_stops').reindex(range(0, 11), fill_value=0).reset_index()
    imstp2 = imstp2.set_index('all_stops').reindex(range(0, 11), fill_value=0).reset_index()
    s_all = pd.DataFrame() 
    s_all['% of (' + name1 + ')'] = list(imstp1['percentage'])
    s_all['% of (' + name2 + ')'] = list(imstp2['percentage'])
    s_all['# Stops in tour'] = range(0, 11)
    s_all = s_all.set_index('# Stops in tour')
    s_all = get_differences(s_all, '% of (' + name1 + ')', '% of (' + name2 + ')', 3)
    s_all = s_all[["% of (DaysimOutputs)", f'% of ({survey_year}Survey)', f'Difference (% of ({name1}) - % of ({name2}))']]
    # display table
    display(s_all.style.format({
        f'% of ({name1})': '{:,.2f}%',
        f'% of ({name2})': '{:,.2f}%',
        f'Difference (% of ({name1}) - % of ({name2}))': '{:,.2f}%',
    }))
    # bar plot
    fig = px.bar(
        s_all.reset_index(),
        x='# Stops in tour',
        y=[f'% of ({name1})', f'% of ({name2})'],
        barmode='group',
        title=f'Number of Stops per Tour (All Purposes) - {tag}'
    )
    fig.update_layout(yaxis_title='Percentage of Tours', 
                      xaxis_title='Number of Stops', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()
    

In [ ]:
num_stops_all_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
num_stops_all_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip per Person

In [ ]:
from collections import OrderedDict

def trip_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trip per person
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    #Total trips per person
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    atp1 = get_total(data1['Trip']['trexpfac']) / Person_1_total
    atp2 = get_total(data2['Trip']['trexpfac']) / Person_2_total
    travdist_data1 = data1['Trip'][(data1['Trip']['travdist']>0)]
    travdist_data1 = travdist_data1[(travdist_data1['travdist']<200)].copy(deep=True)
    travdist_data2 = data2['Trip'].query('travdist > 0 and travdist < 200').copy(deep=True)
    atl1 = weighted_average(travdist_data1, 'travdist', 'trexpfac')
    atl2 = weighted_average(travdist_data2, 'travdist', 'trexpfac')
    ttp1 = [atp1, atl1]
    ttp2 = [atp2, atl2]
    label = ['Average Trips Per Person', 'Average Trip Length']
    items = OrderedDict((('', label), (name1, ttp1), (name2, ttp2)))
    ttp = pd.DataFrame.from_dict(items)
    ttp = ttp.set_index('')
    ttp = get_differences(ttp, name1, name2, 2)
    # table
        # format decimals per row
    row_decimals = {'Average Trips Per Person': 2,
                    'Average Trip Length': 2,
                    }
    styler = ttp.style
    for row_label in ttp.index:
        decimals = row_decimals.get(row_label, 1)
        styler = styler.format({'DaysimOutputs': f'{{:.{decimals}f}}',
                                f'{survey_year}Survey': f'{{:.{decimals}f}}',
                                f'Difference (DaysimOutputs - {survey_year}Survey)': f'{{:.{decimals}f}}',
                                f'% Difference (DaysimOutputs - {survey_year}Survey)': '{:.1f}%'},
                                subset=pd.IndexSlice[[row_label], :])
    display(styler)

In [ ]:
trip_per_person(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trip_per_person(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trips per Person by Purpose

In [ ]:
def trip_rate_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trip Rates by Purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    trp1 = data1['Trip'][['dpurp', 'trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_1_total
    trp2 = data2['Trip'][['dpurp', 'trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_2_total
    trp = pd.DataFrame()
    trp['Trips per Person (' + name1 + ')'] = trp1
    trp['Trips per Person (' + name2 + ')'] = trp2
    trp = get_differences(trp, 'Trips per Person (' + name1 + ')', 'Trips per Person (' + name2 + ')', 2)
    trp = recode_index(trp, 'dpurp', 'Destination Purpose')
    trp = trp.loc[pdpurp_cat.values()]
    # table
    display(trp.style.format({
        f'Trips per Person ({name1})': '{:,.2f}',
        f'Trips per Person ({name2})': '{:,.2f}',
        f'Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.2f}',
        f'% Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}%'
    }))
    fig = px.bar(
        trp.reset_index(),
        x='Destination Purpose',
        y=[f'Trips per Person ({name1})', f'Trips per Person ({name2})'],
        barmode='group',
        title=f'Trip Rates by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', 
                      xaxis_title='Destination Purpose',
                      xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    
    fig.show()

In [ ]:
trip_rate_by_purp(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trip_rate_by_purp(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trips per Person by Mode

In [ ]:
def trip_rate_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trip Rates by Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    trp1 = data1['Trip'][['mode', 'trexpfac']].groupby('mode').sum()['trexpfac'] / Person_1_total
    trp2 = data2['Trip'][['mode', 'trexpfac']].groupby('mode').sum()['trexpfac'] / Person_2_total
    trp = pd.DataFrame()
    trp['Trips per Person (' + name1 + ')'] = trp1
    trp['Trips per Person (' + name2 + ')'] = trp2
    trp = get_differences(trp, 'Trips per Person (' + name1 + ')', 'Trips per Person (' + name2 + ')', 2)
    trp = recode_index(trp, 'mode', 'Destination Mode')
    trp = trp.loc[trip_mode_cat.values()]
    # table
    display(trp.style.format({
        f'Trips per Person ({name1})': '{:,.2f}',
        f'Trips per Person ({name2})': '{:,.2f}',
        f'Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.2f}',
        f'% Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}%'
    }))
    fig = px.bar(
        trp.reset_index(),
        x='Destination Mode',
        y=[f'Trips per Person ({name1})', f'Trips per Person ({name2})'],
        barmode='group',
        title=f'Trip Rates by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', 
                      xaxis_title='Number of Stops', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trip_rate_by_mode(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trip_rate_by_mode(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trip Distance by Purpose

In [ ]:
def trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose') 
    atl = atl.loc[pdpurp_cat.values()] 
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.2f}',
        f'Average Trip Length ({name2})': '{:,.2f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', 
                      xaxis_title='Trip Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_distance_by_purp(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trips_distance_by_purp(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trip Distance by Mode

In [ ]:
def trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Purpose')  
    atl = atl.loc[trip_mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.2f}',
        f'Average Trip Length ({name2})': '{:,.2f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', 
                      xaxis_title='Trip Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_distance_by_mode(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trips_distance_by_mode(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trips per Tour by Tour Purpose

In [ ]:
def trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                                   feature='pdpurp', feature_label='Purpose', feature_order=[]):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    
    ##Count number of trips on each tour by tour purpose``
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    notrips1 = tourtrip1[['hhno', 'pno', 'tour', 'day', 'trexpfac']].groupby(['hhno', 'pno', 'tour', 'day']).count()['trexpfac']
    notrips2 = tourtrip2[['hhno', 'pno', 'tour', 'day', 'trexpfac']].groupby(['hhno', 'pno', 'tour', 'day']).count()['trexpfac']
    notrips1 = notrips1.reset_index()
    notrips2 = notrips2.reset_index()
    notrips1 = notrips1.rename(columns = {'trexpfac':'notrips'})
    notrips2 = notrips2.rename(columns = {'trexpfac':'notrips'})

    #Merge number of trips with the tour file 
    toursnotrips1 = pd.merge(tour_ok_1[['toexpfac', 'pdpurp', 'hhno', 'pno', 'tour', 'tmodetp']], notrips1, on = ['hhno', 'pno', 'tour'])
    toursnotrips2 = pd.merge(tour_ok_2[['toexpfac', 'pdpurp', 'hhno', 'pno', 'tour', 'tmodetp']], notrips2, on = ['hhno', 'pno', 'tour'])

    #Get the average number of trips per tour
    tourtotal1 = weighted_average(toursnotrips1, 'notrips', 'toexpfac', feature)
    tourtotal2 = weighted_average(toursnotrips2, 'notrips', 'toexpfac', feature)

    #Create data frame
    nttp1 = pd.DataFrame.from_dict({'Avg # Trips/Tour (' + name1 + ')': tourtotal1})
    nttp2 = pd.DataFrame.from_dict({'Avg # Trips/Tour (' + name2 + ')': tourtotal2})
    nttp = pd.merge(nttp1, nttp2, 'outer', left_index = True, right_index = True)
    nttp = get_differences(nttp, 'Avg # Trips/Tour (' + name1 + ')', 'Avg # Trips/Tour (' + name2 + ')', 2)
    nttp = recode_index(nttp, feature, f'Tour {feature_label}')
    nttp = nttp.loc[feature_order]

    # display table
    display(nttp.style.format({
        f'Avg # Trips/Tour ({name1})': '{:,.2f}',
        f'Avg # Trips/Tour ({name2})': '{:,.2f}',
        f'Difference (Avg # Trips/Tour ({name1}) - Avg # Trips/Tour ({name2}))': '{:,.2f}',
        f'% Difference (Avg # Trips/Tour ({name1}) - Avg # Trips/Tour ({name2}))': '{:,.1f}%',
    }))
    # bar plot
    fig = px.bar(
        nttp.reset_index(),
        x=f'Tour {feature_label}',
        y=[f'Avg # Trips/Tour ({name1})', f'Avg # Trips/Tour ({name2})'],
        barmode='group',
        title=f'Average Number of Trips per Tour by Tour {feature_label} ({tag})'
    )
    fig.update_layout(yaxis_title='Average # Trips/Tour', 
                      xaxis_title=f'Tour {feature_label}',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                               feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

In [ ]:
trips_per_tour_by_tour_feature(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                               feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

## Trips per Tour by Tour Mode

In [ ]:
trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                               feature='tmodetp', feature_label='Mode', feature_order=mode_cat.values())

In [ ]:
trips_per_tour_by_tour_feature(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                               feature='tmodetp', feature_label='Mode', feature_order=mode_cat.values())